# simulate_causal

> 对应代码：`EconometricsCode/code_in_notes/Chap.26/simulate_causal.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.26`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.26")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们生成 100 期数据，并同时构造两个时间下标：t1 为正序（1 到 N），t2 为倒序（N 到 1）。稍后我们会分别按 t1 和 t2 声明时间序列，从而可以让''滞后算子 L''作用在相反的时间方向上，用以演示非因果（non-causal）的 AR 过程。

In [ ]:
%%stata
clear
set obs 100
set seed 505
gen t1=_n
gen t2=_N-_n+1
gen e=rnormal()

按正序 t1 声明时间序列，然后构造一个非因果 AR(1) 的 MA(∞) 表示。我们要模拟的过程是 y_t = 2 y_{t+1} + e_t（AR 系数 2 不满足因果性条件 |φ|<1），其因果表示不存在，但可以写成对未来冲击的加权平均 y_t = -Σ (1/2)^j e_{t+j}。这里我们截断到 5 阶来近似：先把 tsset 设为 t1，用 L.e 表示 e 的''过去''值，等式右边取负、对应前向加权的符号。

In [ ]:
%%stata
tsset t1
gen y=0
replace y=-1*((1/2)*L.e+(1/2^2)*L2.e+(1/2^3)*L3.e+ ///
(1/2^4)*L4.e+(1/2^5)*L5.e) if t1>5

接着我们把 tsset 切换为倒序 t2。在 t2 的时间方向上，''滞后'' L.y 实际对应原时间轴上的 y_{t+1}。我们计算 er = y_t - 2 y_{t+1}，理论上它应该恰好还原出原始冲击 e_t（在我们截断近似的精度内）。这验证了非因果 AR 过程可以表示为对未来冲击的加权和。

In [ ]:
%%stata
tsset t2
gen er=y-2*L.y

画出 y 的时间序列图：非因果 AR 过程同样平稳，序列外观与因果 AR 难以区分，说明仅凭时间序列图无法判断因果方向。

In [ ]:
%%stata
tsline y

把真实冲击 e 与还原出的 er 画在一起（去掉末端受截断影响的观测），两者几乎重合，验证了我们的构造。

In [ ]:
%%stata
tsline e er if t2<95

绘制 y 的 ACF。非因果 AR(1)（系数 2）与因果 AR(1)（系数 1/2）拥有完全相同的自相关函数，这说明 ACF 无法区分因果与非因果表示——这正是我们要求因果性（|φ|<1）以保证可识别性的原因。

In [ ]:
%%stata
ac y

最后我们对照模拟一个标准的因果 AR(1)（系数 2 按 t2 方向递推，相当于沿倒序方向的爆炸性过程），画出序列图作对比，帮助理解平稳性条件在两个时间方向上的对称性。

In [ ]:
%%stata
gen y2=0
replace y2=2*L.y2+e if t2>1
tsline y2